# Masterclass ETL y Data Quality - LRA

**Recurso compartido por el equipo de Arquitectura | LRACO Group | lraco.group@bbva.com**

> Este laboratorio aterriza ETL y calidad de datos como algo verificable: perfilado, reglas, limpieza, evidencia y una salida final que ya se puede consumir con mas confianza.

**Cuando conviene abrirlo:** cuando quieres explicar pipelines de datos mas alla de la transformacion tecnica y enfocarte en calidad, consistencia y criterio operativo.

**Que te deberia dejar:** una base clara para perfilar datos, escribir reglas simples, medir calidad y cerrar un flujo ETL con evidencia visible.


In [ ]:
# --- Setup minimo para el lab de ETL/Data Quality ---
import subprocess
import sys

for pkg in ['pandas', 'matplotlib']:
    try:
        __import__(pkg)
        print(f'{pkg} ya disponible')
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])


In [ ]:
# Bootstrap: descargar recursos desde GitHub raw
from pathlib import Path
from urllib.request import urlretrieve
import base64

BASE = base64.b64decode(
    'aHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL1phbWlyUGluZWRhL3NwYXJrX2NvbGFiX3BhY2thZ2UvbWFpbg=='
).decode()

FILES = {
    'content/fact_sales.csv': f'{BASE}/content/fact_sales.csv',
    'shared/lra_lab_visuals.py': f'{BASE}/shared/lra_lab_visuals.py',
}

downloaded = []
skipped = []

for rel_path, url in FILES.items():
    path = Path(rel_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        skipped.append(rel_path)
        continue
    urlretrieve(url, path)
    downloaded.append(rel_path)

print(f'Recursos descargados: {len(downloaded)}')
print(f'Recursos ya presentes: {len(skipped)}')


In [ ]:
# --- Helpers visuales compartidos de la coleccion ---
try:
    from shared.lra_lab_visuals import (
        callout,
        checkpoint,
        closing_card,
        hero_header,
        section_divider,
        team_channels_card,
    )
    print('Helpers visuales importados desde shared.lra_lab_visuals')
except Exception:
    from IPython.display import HTML, display

    def callout(kind, title, message):
        palette = {
            'tip': ('#fff7ed', '#ea580c', '#7c2d12'),
            'summary': ('#eff6ff', '#2563eb', '#1e3a8a'),
            'error': ('#fef2f2', '#dc2626', '#7f1d1d'),
            'prod': ('#f0fdf4', '#16a34a', '#14532d'),
        }
        bg, border, text = palette.get(kind, palette['summary'])
        display(HTML(f"""
        <div style='margin:12px 0;font-family:Segoe UI,sans-serif;'>
          <div style='background:{bg};border:1px solid {border};border-left:6px solid {border};border-radius:16px;padding:14px 16px;'>
            <div style='font-size:12px;text-transform:uppercase;letter-spacing:.08em;color:{border};font-weight:700;margin-bottom:8px;'>{title}</div>
            <div style='font-size:14px;line-height:1.65;color:{text};'>{message}</div>
          </div>
        </div>
        """))

    def checkpoint(number, completed, total):
        progress = 0 if total <= 0 else round((completed / total) * 100)
        display(HTML(f"""
        <div style='margin:16px 0;font-family:Segoe UI,sans-serif;'>
          <div style='background:#f8fafc;border:1px solid #e2e8f0;border-radius:16px;padding:14px 16px;'>
            <div style='font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;color:#475569;'>Checkpoint - Parte {number} completada</div>
            <div style='margin-top:8px;font-size:14px;color:#0f172a;'>Progreso visible: {completed} de {total} bloques completados.</div>
            <div style='margin-top:10px;height:10px;background:#e2e8f0;border-radius:999px;overflow:hidden;'><div style='height:10px;width:{progress}%;background:linear-gradient(90deg,#2563eb,#14b8a6);'></div></div>
          </div>
        </div>
        """))

    def section_divider(part, title, subtitle, accent='#1d4ed8'):
        display(HTML(f"""
        <div style='margin:26px 0 14px;font-family:Segoe UI,sans-serif;'>
          <div style='font-size:12px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:{accent};margin-bottom:6px;'>Parte {part}</div>
          <div style='font-size:30px;font-weight:800;color:#0f172a;line-height:1.08;'>{title}</div>
          <div style='font-size:15px;line-height:1.7;color:#475569;max-width:860px;margin-top:8px;'>{subtitle}</div>
        </div>
        """))

    def hero_header(theme, title, description, pills):
        pills_html = ''.join([f"<span style='background:rgba(240,253,244,.12);border:1px solid rgba(240,253,244,.16);padding:9px 12px;border-radius:999px;font-size:13px;'>{pill}</span>" for pill in pills])
        display(HTML(f"""
        <div style='font-family:Segoe UI,sans-serif; margin:18px 0 12px;'>
          <div style='background:linear-gradient(135deg,#052e16 0%,#166534 48%,#65a30d 100%);color:#f0fdf4;padding:30px 28px;border-radius:24px;box-shadow:0 20px 42px rgba(15,23,42,.18);'>
            <div style='display:inline-block;background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.18);padding:7px 12px;border-radius:999px;font-size:12px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;'>Masterclass LRA - ETL y Data Quality</div>
            <h1 style='margin:14px 0 10px;font-size:34px;line-height:1.06;'>{title}</h1>
            <p style='margin:0;font-size:15px;line-height:1.7;color:#dcfce7;max-width:760px;'>{description}</p>
            <div style='display:flex;flex-wrap:wrap;gap:10px;margin-top:18px;'>{pills_html}</div>
          </div>
        </div>
        """))

    def closing_card(accent_gradient, bullets):
        bullets_html = ''.join([f"<div style='background:rgba(255,255,255,.10);border-radius:14px;padding:12px 14px;'>{item}</div>" for item in bullets])
        display(HTML(f"""
        <div style='margin:14px 0 16px;font-family:Segoe UI,sans-serif;'>
          <div style='background:{accent_gradient}; color:#f8fafc; border-radius:18px;padding:18px 20px;'>
            <div style='font-size:12px;text-transform:uppercase;letter-spacing:.14em;opacity:.82;'>Cierre de aprendizaje</div>
            <div style='font-size:24px;font-weight:800;margin:8px 0 10px;'>Que deberias llevarte de esta masterclass</div>
            <div style='display:grid;grid-template-columns:repeat(2, minmax(0, 1fr));gap:10px;margin-top:10px;'>{bullets_html}</div>
          </div>
        </div>
        """))

    def team_channels_card(title, accent, bg, border):
        display(HTML(f"""
        <div style='margin:12px 0 12px;font-family:Segoe UI,sans-serif;'>
          <div style='background:{bg}; border:1px solid {border}; border-left:6px solid {accent};border-radius:16px; padding:16px 18px;'>
            <div style='font-size:12px;text-transform:uppercase;letter-spacing:.1em;color:{accent};font-weight:700;margin-bottom:8px;'>Canales del equipo</div>
            <div style='font-size:16px;font-weight:700;color:#0f172a;margin-bottom:8px;'>{title}</div>
            <div style='font-size:14px;line-height:1.65;color:#334155;'>
              <div><strong>Google Site:</strong> reemplaza aqui el enlace oficial del equipo.</div>
              <div><strong>TeamSpace:</strong> reemplaza aqui el enlace oficial del espacio de colaboracion.</div>
              <div><strong>Correo de soporte:</strong> lraco.group@bbva.com</div>
              <div><strong>Responsable del recurso:</strong> equipo de Arquitectura LRA</div>
            </div>
          </div>
        </div>
        """))


In [ ]:
# --- Header principal del laboratorio ---
hero_header(
    theme='duckdb',
    title='Pipelines con evidencia de calidad y no solo transformaciones',
    description='La meta es recorrer un flujo ETL pequeno pero serio: perfilar, declarar reglas, limpiar, medir impacto y producir una salida final que ya se pueda consumir con mas confianza.',
    pills=['Perfilado y reglas', 'Limpieza visible', 'Salida validada'],
)


## Accesos y soporte LRA

Este recurso esta pensado para publicarse y compartirse como material vivo del equipo de Arquitectura. Si alguien llega desde Google Site o TeamSpace, este bloque le da contexto rapido y una ruta clara para ejecutar el lab.

### Sugerencia de uso
- abre este notebook si hoy quieres discutir calidad de datos con evidencia y no solo con intuicion
- ejecuta desde arriba para que queden cargados los helpers, dataframes y reglas
- si facilitas la sesion, pide una hipotesis sobre donde estaran los problemas antes de perfilar

## Checklist de publicacion

Antes de compartir este notebook, revisa esto:
- que la ruta a `content/fact_sales.csv` siga funcionando
- que las reglas de calidad sean legibles y faciles de extender
- que los graficos se vean bien en Colab
- que la salida final quede escrita dentro de `output/`

## Ruta practica para developers nuevos

1. perfila primero el dataset para entender el estado actual
2. declara luego reglas simples y mide cuantos registros fallan
3. limpia de forma explicita y compara antes vs despues
4. exporta una salida validada y cierra con una lectura de criterio

## Puente ETL/Data Quality -> Spark y DuckDB

Este lab conecta muy bien con la ruta de datos del equipo: Spark puede transformar a escala, DuckDB puede explorar archivos muy rapido y este notebook ayuda a razonar donde y como poner controles de calidad dentro del flujo.


In [ ]:
section_divider(1, 'Fundamentos Guiados de ETL y Calidad', 'Antes de limpiar, perfilamos. Antes de confiar, medimos.', accent='#15803d')


In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('content/fact_sales.csv')
raw_df = pd.read_csv(DATA_PATH)

record_id_col = 'order_id' if 'order_id' in raw_df.columns else 'sale_id'
if 'sale_date' not in raw_df.columns:
    raw_df['sale_date'] = pd.date_range('2026-01-01', periods=len(raw_df), freq='D').astype(str)
    print('Columna sale_date no venia en el CSV base. Se genero una fecha controlada para el laboratorio.')

print('Filas:', len(raw_df))
print('Columnas:', list(raw_df.columns))
print('Columna identificadora usada para unicidad:', record_id_col)
print(raw_df.head())


## 1.1 Modelo mental rapido

En este laboratorio vamos a tratar calidad de datos como una cadena simple:
- **profiling**: entender el estado actual del dataset
- **rules**: declarar que significa "dato sano" para este caso
- **cleansing**: decidir como corregir o excluir
- **evidence**: mostrar el impacto antes y despues

La calidad no aparece al final del pipeline por accidente. Se dise?a y se mide.


In [ ]:
profile = pd.DataFrame({
    'column': raw_df.columns,
    'nulls': raw_df.isna().sum().values,
    'null_pct': (raw_df.isna().mean() * 100).round(2).values,
    'distinct': [raw_df[c].nunique(dropna=True) for c in raw_df.columns],
})
profile


In [ ]:
import matplotlib.pyplot as plt

plot_profile = profile[profile['nulls'] > 0].copy()
if plot_profile.empty:
    print('No se encontraron nulos para graficar en este dataset base.')
else:
    plt.figure(figsize=(8, 4))
    plt.bar(plot_profile['column'], plot_profile['nulls'], color='#16a34a')
    plt.title('Nulos detectados por columna')
    plt.ylabel('Cantidad de nulos')
    plt.xticks(rotation=20)
    for idx, value in enumerate(plot_profile['nulls']):
        plt.text(idx, value + 0.05, str(int(value)), ha='center', fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.25)
    plt.show()


In [ ]:
callout('summary', 'Micro-resumen', 'Un profiling simple ya te obliga a dejar de hablar de calidad en abstracto: ahora puedes mostrar columnas, nulos y cardinalidad con evidencia.')


In [ ]:
checkpoint(1, 3, 10)


In [ ]:
section_divider(2, 'Laboratorio de Reglas de Calidad', 'Declaramos fallas esperables y medimos cuantas filas rompen cada criterio', accent='#2563eb')


In [ ]:
dirty_df = raw_df.copy()

# Inyectamos problemas controlados para que el laboratorio tenga fallas visibles.
dirty_df.loc[2, 'amount'] = -50
dirty_df.loc[4, 'customer_id'] = None
dirty_df.loc[7, 'sale_date'] = None
dirty_df.loc[len(dirty_df)] = dirty_df.iloc[0]

dirty_df.tail()


In [ ]:
rules = {
    'customer_id_present': dirty_df['customer_id'].notna(),
    'sale_date_present': dirty_df['sale_date'].notna(),
    'amount_positive': dirty_df['amount'] > 0,
    f'{record_id_col}_unique': ~dirty_df[record_id_col].duplicated(keep=False),
}

quality_report = pd.DataFrame([
    {
        'rule': name,
        'passed_rows': int(mask.sum()),
        'failed_rows': int((~mask).sum()),
        'failed_pct': round((~mask).mean() * 100, 2),
    }
    for name, mask in rules.items()
]).sort_values('failed_rows', ascending=False)

quality_report


In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(quality_report['rule'], quality_report['failed_rows'], color='#2563eb')
plt.title('Filas que fallan por regla de calidad')
plt.xlabel('Cantidad de filas fallidas')
for idx, value in enumerate(quality_report['failed_rows']):
    plt.text(value + 0.05, idx, str(int(value)), va='center', fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.25)
plt.show()


In [ ]:
callout('error', 'Error comun', 'Hablar de calidad como si fuera una sola metrica. En la practica conviene separar completitud, validez y unicidad para leer mejor el problema.')


In [ ]:
checkpoint(2, 6, 10)


In [ ]:
section_divider(3, 'Limpieza y Evidencia', 'Aplicamos correcciones explicitas y medimos el impacto antes vs despues', accent='#7c3aed')


In [ ]:
clean_df = dirty_df.copy()
clean_df = clean_df.dropna(subset=['customer_id', 'sale_date'])
clean_df = clean_df[clean_df['amount'] > 0]
clean_df = clean_df.drop_duplicates(subset=[record_id_col])
clean_df = clean_df.reset_index(drop=True)

summary_before_after = pd.DataFrame({
    'estado': ['antes', 'despues'],
    'rows': [len(dirty_df), len(clean_df)],
    'null_customer_id': [int(dirty_df['customer_id'].isna().sum()), int(clean_df['customer_id'].isna().sum())],
    'null_sale_date': [int(dirty_df['sale_date'].isna().sum()), int(clean_df['sale_date'].isna().sum())],
    'non_positive_amount': [int((dirty_df['amount'] <= 0).sum()), int((clean_df['amount'] <= 0).sum())],
    f'duplicate_{record_id_col}': [int(dirty_df[record_id_col].duplicated().sum()), int(clean_df[record_id_col].duplicated().sum())],
})
summary_before_after


In [ ]:
metrics = ['null_customer_id', 'null_sale_date', 'non_positive_amount', f'duplicate_{record_id_col}']
plot_df = summary_before_after.set_index('estado')[metrics].T
plot_df.plot(kind='bar', figsize=(9, 4), color=['#f97316', '#16a34a'])
plt.title('Antes vs despues de la limpieza')
plt.ylabel('Cantidad de incidencias')
plt.xticks(rotation=20)
plt.grid(axis='y', linestyle='--', alpha=0.25)
plt.show()


In [ ]:
callout('prod', 'Llevalo a produccion', 'En un pipeline real, estas reglas no deberian vivir solo en un notebook. Lo ideal es versionarlas, medir tendencias y decidir si una falla bloquea o solo alerta.')


In [ ]:
checkpoint(3, 8, 10)


In [ ]:
section_divider(4, 'Salida Validada', 'Exportamos una salida limpia y cerramos con una lectura operativa', accent='#ea580c')


In [ ]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
validated_path = output_dir / 'fact_sales_validated.csv'
clean_df.to_csv(validated_path, index=False)
print('Salida validada escrita en:', validated_path)
print(clean_df.head())


## Si continuas en DuckDB o Spark

Despues de ejecutar esta exportacion, puedes reutilizar la salida limpia en otros labs de la coleccion:

- `Masterclass_DuckDB.ipynb`: crea una vista sobre `output/fact_sales_validated.csv` para explorar el dataset ya validado.
- `Masterclass_Spark.ipynb`: lee `output/fact_sales_validated.csv` como input confiable para otra transformacion o un handoff posterior.

### Idea del patron
- ETL/Data Quality deja evidencia y una salida mas sana.
- DuckDB la puede inspeccionar rapido.
- Spark la puede usar como siguiente etapa del pipeline.


## 4.1 Caso de cierre

Si este fuera un pipeline del equipo, esta salida validada ya podria pasar a una siguiente capa de consumo con una confianza mejor documentada.

La pregunta final no es solo "quedo limpio". La pregunta final es:
- que reglas bloquearian el pipeline
- cuales solo generarian alerta
- quien deberia enterarse
- y donde dejas evidencia de la decision


In [ ]:
final_checks = {
    'rows_validated': len(clean_df),
    'customer_id_nulls': int(clean_df['customer_id'].isna().sum()),
    'sale_date_nulls': int(clean_df['sale_date'].isna().sum()),
    'non_positive_amount': int((clean_df['amount'] <= 0).sum()),
    f'duplicate_{record_id_col}': int(clean_df[record_id_col].duplicated().sum()),
}
final_checks


## 4.2 Preguntas de cierre

Antes de irte, intenta responder esto:
1. Que reglas deberian bloquear el pipeline y cuales solo alertar?
2. Que evidencia dejarias para que otro equipo entienda por que se excluyeron filas?
3. En que parte pondrias estas validaciones: antes, durante o despues de la transformacion principal?
4. Como conectarias este control con Spark o DuckDB en un flujo real?


In [ ]:
closing_card(
    'linear-gradient(135deg,#14532d,#16a34a)',
    [
        'La calidad de datos mejora cuando dejas reglas explicitas y evidencia visible, no solo intuicion.',
        'Profiling, validacion y limpieza forman una cadena peque?a pero muy poderosa para ETL.',
        'Antes vs despues es una de las mejores formas de explicar impacto tecnico a otras personas del equipo.',
        'El siguiente paso natural es llevar estas reglas a un pipeline o motor real con monitoreo.'
    ]
)


In [ ]:
team_channels_card('Si quieres llevar este laboratorio a un pipeline mas real', '#16a34a', '#f0fdf4', '#86efac')


## Cierre del equipo

Gracias por dedicar tiempo a este laboratorio.

Desde el equipo de Arquitectura LRA buscamos que este recurso ayude a hablar de ETL y calidad con mas criterio: que problema detectaste, que regla aplicaste, que impacto tuvo y que decision operativa se desprende de eso.

## Conclusiones esperadas

Si este notebook cumplio su objetivo, al final deberias quedarte con estas ideas:
- ETL no es solo mover datos: tambien es decidir que tan confiable queda la salida
- la calidad mejora cuando defines reglas simples y legibles
- comparar antes vs despues ayuda mucho a comunicar impacto
- este tipo de controles conecta muy bien con Spark, DuckDB o pipelines mas grandes del equipo
